# ABCA4 Advanced End-to-End Pipeline

**Canonical, single-pass, leakage-safe pipeline for ABCA4 variant pathogenicity classification.**

### What this notebook does
1. Loads your annotated mutations CSV and fetches the P78363 wild-type sequence from UniProt.
2. Computes ESM-2 delta embeddings (±20 aa window, 650 M parameter transformer) via HuggingFace – cached locally.
3. Reduces 1,280-dimensional ESM features to 50 PCA components (fit on train, transform VUS – zero leakage).
4. Annotates biologically important domains (TMD, NBD, ECD1, ECD2, Gate residues).
5. **Streams AlphaMissense scores directly from DeepMind** (`storage.googleapis.com/dm_alphamissense`) on first run; cached thereafter.
6. Runs a leakage-safe 5-fold `StratifiedGroupKFold` CV (grouped by residue position) with in-fold SHAP feature selection, `BorderlineSMOTE`, per-fold isotonic calibration, and MCC threshold optimisation.
7. Fits a final production model and exports all artefacts.
8. Benchmarks the enriched model against AlphaMissense alone (AUROC, MCC, Brier).
9. Predicts pathogenicity for all VUS variants.
10. Validates against the 2024 gold-standard reclassification set.
11. Runs a leakage + robustness audit.

### Required files (place alongside this notebook)
| File | Description |
|---|---|
| `ABCA4_mutations_annotated_with_features.csv` **or** `ABCA4_mutations_annotated_with_features copy.csv` | Your primary annotated variants dataset |

All other files are fetched / generated automatically.

### Recommended runtime
Google Colab with a **T4 GPU** (or A100). ESM-2 embeddings and the full CV loop are heavy.


## Section 1 — Imports, Constants & Paths

In [ ]:
import io
import os
import re
import json
import gzip
import pickle
import warnings
from collections import Counter
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
import requests
import shap
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss, matthews_corrcoef
from sklearn.isotonic import IsotonicRegression
from imblearn.over_sampling import BorderlineSMOTE
from transformers import AutoTokenizer, AutoModel

try:
    from sklearn.frozen import FrozenEstimator
except ImportError:
    class FrozenEstimator:
        """Prevents sklearn from re-fitting a pre-trained estimator during calibration."""
        def __init__(self, estimator):
            self.estimator = estimator
        def fit(self, X, y=None):
            return self
        def predict_proba(self, X):
            return self.estimator.predict_proba(X)
        def predict(self, X):
            return self.estimator.predict(X)

# ── Random seeds ──────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# ── Paths ─────────────────────────────────────────────────────────────────
ROOT = Path.cwd()

# Auto-detect the mutations CSV (handles both filename variants)
_candidates = [
    ROOT / 'ABCA4_mutations_annotated_with_features.csv',
    ROOT / 'ABCA4_mutations_annotated_with_features copy.csv',
]
DATA_PATH = next((p for p in _candidates if p.exists()), _candidates[0])
print(f'Using data file: {DATA_PATH}')

WT_CACHE_PATH          = ROOT / 'abca4_wt_sequence.txt'
ESM_CACHE_PATH         = ROOT / 'esm_cache.pkl'
ALPHAMISSENSE_PATH     = ROOT / 'ABCA4_alphamissense_scores.csv'
FEATURE_STABILITY_PATH = ROOT / 'feature_stability.csv'
MODEL_PATH             = ROOT / 'abca4_binary_model.json'
CALIBRATOR_PATH        = ROOT / 'isotonic_calibrator.pkl'
FEATURES_PATH          = ROOT / 'features.json'
THRESHOLD_PATH         = ROOT / 'threshold.json'
OOF_PATH               = ROOT / 'oof_predictions.csv'
VUS_PRED_PATH          = ROOT / 'vus_predictions.csv'

# ── External URLs ─────────────────────────────────────────────────────────
UNIPROT_FASTA_URL = 'https://rest.uniprot.org/uniprotkb/P78363.fasta'
ESM_MODEL_NAME    = 'facebook/esm2_t33_650M_UR50D'
ALPHAMISSENSE_URL = 'https://storage.googleapis.com/dm_alphamissense/AlphaMissense_aa_substitutions.tsv.gz'
ALPHAMISSENSE_UID = 'P78363'

# ── ESM constants ─────────────────────────────────────────────────────────
WINDOW    = 20
EMBED_DIM = 1280

# ── Label string sets ─────────────────────────────────────────────────────
BENIGN_STRINGS     = {'benign', 'likely benign'}
PATHOGENIC_STRINGS = {'pathogenic', 'likely pathogenic'}
AMBIGUOUS_STRINGS  = {
    'vus', 'variant of uncertain significance', 'uncertain significance',
    'uncertain', 'ambiguous', 'conflicting',
    'conflicting interpretations of pathogenicity', 'unknown', 'not provided',
}

METADATA_DROP = [
    'Variant', 'Significance', 'Significance_norm', 'Source',
    'Annotation', 'binary_label', 'is_vus',
]


## Section 2 — Helper Functions

All utility functions are defined exactly once here. Later cells call them by name without re-defining them.

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────

def engineer_contact_aggregates(df_: pd.DataFrame) -> pd.DataFrame:
    """Add mean/max/sum aggregate columns for HH, PP, HP contact shells."""
    out = df_.copy()
    for label, prefix in [('HH', 'HH:'), ('PP', 'PP:'), ('HP', 'HP:')]:
        cols = [c for c in df_.columns if c.startswith(prefix)]
        if cols:
            out[f'{label}_mean'] = df_[cols].mean(axis=1)
            out[f'{label}_max']  = df_[cols].max(axis=1)
            out[f'{label}_sum']  = df_[cols].sum(axis=1)
    return out


def add_domain_features(df_: pd.DataFrame) -> pd.DataFrame:
    """Annotate TMD, NBD, ECD1, ECD2, and Gate residues for ABCA4 (P78363).

    Ranges from Xie et al. 2021 cryo-EM structure / AlphaFold2 topology.
    Gate residues (653, 2107) are critical for the ABCA4 transport mechanism.
    """
    out = df_.copy()
    pos = out['Position'].astype(int)

    tmd_ranges = [(20, 40), (641, 781), (1315, 1390), (1715, 1850)]
    out['is_tmd']  = pos.apply(lambda x: int(any(s <= x <= e for s, e in tmd_ranges)))
    out['is_nbd']  = pos.apply(lambda x: int((884 <= x <= 1150) or (1880 <= x <= 2150)))
    out['is_ecd1'] = pos.apply(lambda x: int(641 <= x <= 1200))
    out['is_ecd2'] = pos.apply(lambda x: int(1395 <= x <= 1680))
    out['is_gate'] = pos.apply(lambda x: int(x in {653, 2107}))

    if 'Consurf_score' in out.columns:
        out['nbd_conservation_impact'] = out['is_nbd'] * out['Consurf_score'].fillna(0)
    if 'Envision_delta_PSIC' in out.columns:
        out['nbd_psic_impact'] = out['is_nbd'] * out['Envision_delta_PSIC'].fillna(0)

    rsa = out['Relative_SASA'].fillna(0.5) if 'Relative_SASA' in out.columns else 0.5
    out['gate_impact_score'] = out['is_gate'] * (1 - rsa)
    return out


def resolve_protected_features(df_: pd.DataFrame) -> list:
    """Return feature column names that must always stay in the model.

    Includes raw contact shell columns (HH:, PP:, HP:), aggregates,
    and biologically critical flags.
    """
    prefixes = ['HH:', 'PP:', 'HP:', 'HH_', 'PP_', 'HP_', 'is_tmd', 'is_nbd',
                'is_ecd', 'is_gate', 'nbd_', 'gate_', 'FunctionalDomain',
                'dHbond', 'VDWClash']
    resolved = set()
    for a in prefixes:
        for c in df_.columns:
            if c.startswith(a) or c == a:
                resolved.add(c)
    return sorted(resolved)


# ── Type pre-processing ───────────────────────────────────────────────────

def preprocess_types(df_: pd.DataFrame, protected_features: list) -> pd.DataFrame:
    """Cast FunctionalDomain to category; coerce remaining object columns to numeric."""
    out = df_.copy()
    if 'FunctionalDomain' in out.columns:
        out['FunctionalDomain'] = (
            out['FunctionalDomain'].astype('string').fillna('Unknown').astype('category')
        )
    for c in out.select_dtypes(include='object').columns:
        if c == 'FunctionalDomain':
            continue
        out[c] = pd.to_numeric(out[c], errors='coerce')
    return out


# ── Imputation ────────────────────────────────────────────────────────────

def fit_imputation_values(df_: pd.DataFrame) -> pd.Series:
    """Return per-column medians for numeric columns. Always fit on training data only."""
    return df_.select_dtypes(include=[np.number]).median()


def apply_imputation(df_: pd.DataFrame, medians: pd.Series, protected_features: list) -> pd.DataFrame:
    """Fill NaN values with pre-computed medians. Leaves category columns untouched."""
    out = df_.copy()
    for col, val in medians.items():
        if col in out.columns and out[col].dtype != 'category':
            out[col] = out[col].fillna(val)
    return out


# ── ESM-2 utilities ───────────────────────────────────────────────────────

_VARIANT_REGEX = re.compile(r'^([A-Za-z\*])([0-9]+)([A-Za-z\*])$')
_ALLOWED_AA    = set('ACDEFGHIKLMNPQRSTVWY')


def parse_variant(v: str):
    """Parse 'R1038W' into (wt_aa, pos_1based, mut_aa) or None."""
    m = _VARIANT_REGEX.match(str(v).strip())
    if not m:
        return None
    return m.group(1).upper(), int(m.group(2)), m.group(3).upper()


def aa_clean(seq: str) -> str:
    return ''.join(ch if ch in _ALLOWED_AA else 'X' for ch in seq)


def get_window(seq: str, pos_1based: int, window: int = WINDOW):
    idx   = pos_1based - 1
    start = max(0, idx - window)
    end   = min(len(seq), idx + window + 1)
    return seq[start:end], idx - start


def mutate_window(window_seq: str, local_idx: int, mut_aa: str) -> str:
    chars = list(window_seq)
    if 0 <= local_idx < len(chars):
        chars[local_idx] = mut_aa
    return ''.join(chars)


def embed_sequence(seq: str, tokenizer, esm_model, dev: torch.device) -> np.ndarray:
    seq = aa_clean(seq)
    enc = tokenizer(seq, return_tensors='pt', add_special_tokens=True)
    enc = {k: v.to(dev) for k, v in enc.items()}
    with torch.no_grad():
        out = esm_model(**enc).last_hidden_state
    tok_emb = out[0, 1:-1, :]   # strip BOS/EOS tokens
    if tok_emb.shape[0] == 0:
        return np.zeros(EMBED_DIM, dtype=np.float32)
    return tok_emb.mean(dim=0).detach().cpu().numpy().astype(np.float32)


print('Helper functions loaded.')


## Section 3 — Data Loading & Label Preparation

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
df_raw['Significance_norm'] = df_raw['Significance'].astype(str).str.strip().str.lower()

df_raw['is_vus'] = df_raw['Significance_norm'].isin(AMBIGUOUS_STRINGS)
df_raw['binary_label'] = np.where(
    df_raw['Significance_norm'].isin(BENIGN_STRINGS),     0,
    np.where(df_raw['Significance_norm'].isin(PATHOGENIC_STRINGS), 1, np.nan),
)

# Engineer contact aggregate features (mean/max/sum per shell type)
df_raw = engineer_contact_aggregates(df_raw)

train_df = df_raw[df_raw['binary_label'].isin([0, 1])].copy()
train_df['binary_label'] = train_df['binary_label'].astype(int)
vus_df   = df_raw[df_raw['is_vus']].copy()

print(f'Dataset shape    : {df_raw.shape}')
print(f'Training rows    : {train_df.shape[0]}  '
      f'(Benign={int((train_df.binary_label == 0).sum())}, '
      f'Pathogenic={int((train_df.binary_label == 1).sum())})')
print(f'VUS rows         : {vus_df.shape[0]}')
print(f'\nSignificance distribution (top 10):')
print(df_raw['Significance_norm'].value_counts(dropna=False).head(10))


## Section 4 — Wild-Type Sequence from UniProt (P78363)

In [ ]:
def load_wt_sequence(cache_path: Path, fasta_url: str) -> str:
    """Fetch the ABCA4 WT FASTA from UniProt REST API. Cached to disk after first run."""
    if cache_path.exists():
        seq = cache_path.read_text().strip().upper()
        if seq:
            print(f'WT sequence loaded from cache ({len(seq)} aa).')
            return seq
    print(f'Fetching WT sequence from UniProt: {fasta_url}')
    resp = requests.get(fasta_url, timeout=60)
    resp.raise_for_status()
    lines = [ln.strip() for ln in resp.text.splitlines()
             if ln.strip() and not ln.startswith('>')]
    seq = ''.join(lines).upper()
    if not seq:
        raise ValueError('Failed to parse sequence from UniProt FASTA response.')
    cache_path.write_text(seq + '\n')
    print(f'WT sequence fetched and cached ({len(seq)} aa) → {cache_path}')
    return seq

wt_sequence = load_wt_sequence(WT_CACHE_PATH, UNIPROT_FASTA_URL)
print(f'First 40 aa: {wt_sequence[:40]}')


## Section 5 — ESM-2 Delta Embeddings

For each variant:
1. Slice a ±20 aa window from the P78363 WT sequence.
2. Run WT and mutant windows through `facebook/esm2_t33_650M_UR50D` (650 M params).
3. Mean-pool the last hidden states → `δ = emb_mut − emb_wt` (1 280-dim).

Results cached to `esm_cache.pkl`.  **Requires a GPU** — first run ~5–20 min.


In [ ]:
device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
hf_token = os.getenv('HF_TOKEN')
print(f'Torch device: {device}')
if not hf_token:
    warnings.warn('HF_TOKEN env var not set – downloads may be rate-limited.')


def build_esm_cache(variants: list, wt_seq: str, cache_path: Path) -> dict:
    """Build or update the ESM-2 delta embedding cache.

    Skips variants already in the cache file to avoid redundant computation.
    """
    cache: dict = {}
    if cache_path.exists():
        with open(cache_path, 'rb') as f:
            cache = pickle.load(f)

    needed = [v for v in variants if v not in cache]
    print(f'ESM cache  hits={len(cache)} | to embed={len(needed)}')

    if needed:
        kwargs    = {'token': hf_token} if hf_token else {}
        tokenizer = AutoTokenizer.from_pretrained(ESM_MODEL_NAME, **kwargs)
        esm_model = AutoModel.from_pretrained(ESM_MODEL_NAME, **kwargs).to(device).eval()

        for i, v in enumerate(needed, 1):
            parsed = parse_variant(v)
            if parsed is None:
                cache[v] = np.zeros(EMBED_DIM, dtype=np.float32)
                continue
            _wt_aa, pos, mut_aa = parsed
            if pos < 1 or pos > len(wt_seq):
                cache[v] = np.zeros(EMBED_DIM, dtype=np.float32)
                continue
            wt_win, ctr_idx = get_window(wt_seq, pos, WINDOW)
            mut_win = mutate_window(wt_win, ctr_idx, mut_aa)
            wt_emb  = embed_sequence(wt_win,  tokenizer, esm_model, device)
            mut_emb = embed_sequence(mut_win, tokenizer, esm_model, device)
            cache[v] = (mut_emb - wt_emb).astype(np.float32)
            if i % 100 == 0 or i == len(needed):
                print(f'  Computed {i}/{len(needed)} ESM deltas ...', end='\r')

        with open(cache_path, 'wb') as f:
            pickle.dump(cache, f)
        print(f'\nESM cache saved → {cache_path}')

    return cache


all_variants = (
    pd.concat([train_df['Variant'], vus_df['Variant']], ignore_index=True)
    .dropna().astype(str).unique().tolist()
)
esm_cache = build_esm_cache(all_variants, wt_sequence, ESM_CACHE_PATH)

esm_cols = [f'esm_delta_{i:04d}' for i in range(EMBED_DIM)]


def attach_esm_features(df_in: pd.DataFrame, cache: dict) -> pd.DataFrame:
    mat = np.vstack([
        cache.get(str(v), np.zeros(EMBED_DIM, dtype=np.float32))
        for v in df_in['Variant'].astype(str)
    ])
    esm_df = pd.DataFrame(mat, columns=esm_cols, index=df_in.index)
    return pd.concat([df_in.copy(), esm_df], axis=1)


train_df = attach_esm_features(train_df, esm_cache)
vus_df   = attach_esm_features(vus_df,   esm_cache)
print(f'train_df with ESM deltas: {train_df.shape}')
print(f'vus_df   with ESM deltas: {vus_df.shape}')


## Section 6 — PCA on ESM-2 Features

Reduces 1 280 ESM delta dimensions to 50 principal components.

**Leakage note:** PCA is `fit` on `train_df` only, then `transform` is applied to `vus_df`.
Within the CV folds the PCA components are shared across fold boundaries — a common,
accepted trade-off since PCA is unsupervised and does not use labels.


In [ ]:
esm_data_train = train_df[esm_cols].values
esm_data_vus   = vus_df[esm_cols].values

pca       = PCA(n_components=50, random_state=RANDOM_STATE)
pca_train = pca.fit_transform(esm_data_train)   # fit on train only
pca_vus   = pca.transform(esm_data_vus)         # apply to VUS — no leakage

pca_cols = [f'esm_pca_{i:02d}' for i in range(50)]

train_df = pd.concat(
    [train_df.drop(columns=esm_cols),
     pd.DataFrame(pca_train, columns=pca_cols, index=train_df.index)],
    axis=1,
)
vus_df = pd.concat(
    [vus_df.drop(columns=esm_cols),
     pd.DataFrame(pca_vus, columns=pca_cols, index=vus_df.index)],
    axis=1,
)

print(f'Reduced ESM features to {len(pca_cols)} PCA components.')
print(f'Explained variance first 10 PCs: '
      f'{pca.explained_variance_ratio_[:10].round(3).tolist()}')


## Section 7 — Domain Annotations

Adds biologically informed binary flags and interaction terms for ABCA4 structural
regions. All domain boundaries from Xie et al. (2021) cryo-EM and AlphaFold2 topology.


In [ ]:
train_df = add_domain_features(train_df)
vus_df   = add_domain_features(vus_df)

# Resolve protected feature list after all engineered columns exist
X_probe            = train_df.drop(columns=METADATA_DROP, errors='ignore')
protected_features = resolve_protected_features(X_probe)
protected_features += pca_cols            # always protect all PCA components
protected_features  = list(dict.fromkeys(protected_features))  # deduplicate

print(f'Domain flags added. Training shape: {train_df.shape}')
print(f'Protected features ({len(protected_features)} total — first 20 shown):')
for f in protected_features[:20]:
    print(f'  {f}')


## Section 8 — AlphaMissense Scores Streamed from DeepMind

This cell streams the official **AlphaMissense_aa_substitutions.tsv.gz** directly
from Google DeepMind's public cloud bucket and extracts only the rows for
UniProt ID `P78363` (ABCA4).

**Why stream instead of using a local file?**
Streaming from the original source guarantees the benchmark scores are exactly
the numbers published in *Cheng et al., Science 2023* — not a pre-filtered
intermediate that could have been accidentally altered or corrupted.

**Memory efficiency:** The decompressor reads line-by-line through `gzip.open(resp.raw)`
so the ~1.5 GB compressed file is never fully loaded into RAM.
The extracted ~3 900 P78363 rows are cached to `ABCA4_alphamissense_scores.csv`
so subsequent runs take seconds instead of minutes.


In [ ]:
def load_alphamissense(cache_path: Path, url: str, uniprot_id: str) -> pd.DataFrame:
    """Stream AlphaMissense scores for *uniprot_id* from DeepMind's public bucket.

    Parameters
    ----------
    cache_path : Path
        Local CSV path; read from cache if it already exists and is non-empty.
    url : str
        URL of ``AlphaMissense_aa_substitutions.tsv.gz``.
    uniprot_id : str
        UniProt accession to filter, e.g. ``'P78363'`` for ABCA4.

    Returns
    -------
    pd.DataFrame with columns ``Variant`` (str) and ``AlphaMissense_score`` (float).
    """
    if cache_path.exists():
        df = pd.read_csv(cache_path)
        if not df.empty:
            print(f'AlphaMissense loaded from local cache '
                  f'({len(df)} variants for {uniprot_id}).')
            return df

    print('Streaming AlphaMissense_aa_substitutions.tsv.gz from Google DeepMind ...')
    print(f'  URL : {url}')
    print(f'  Filtering for uniprot_id == {uniprot_id!r}')
    print('  First run only — file is ~1.5 GB compressed. Expect 3–10 minutes.')

    resp = requests.get(url, stream=True, timeout=600)
    resp.raise_for_status()
    resp.raw.decode_content = True  # ensure urllib3 applies transfer-encoding decode

    rows    = []
    header  = None
    n_lines = 0

    with gzip.open(resp.raw, 'rt') as gz:
        for line in gz:
            line = line.rstrip('\n')

            # The header line starts with '#'
            if line.startswith('#'):
                if header is None:
                    header = line.lstrip('#').split('\t')
                continue
            if header is None:
                continue

            parts = line.split('\t')
            if len(parts) < len(header):
                continue

            n_lines += 1
            row_dict = dict(zip(header, parts))

            if row_dict.get('uniprot_id') == uniprot_id:
                try:
                    rows.append({
                        'Variant':             row_dict['protein_variant'],
                        'AlphaMissense_score': float(row_dict['am_pathogenicity']),
                    })
                except (KeyError, ValueError):
                    pass

            if n_lines % 5_000_000 == 0:
                print(f'  Scanned {n_lines:,} rows | '
                      f'found {len(rows)} {uniprot_id} matches ...', end='\r')

    df = pd.DataFrame(rows).drop_duplicates(subset='Variant').reset_index(drop=True)
    df.to_csv(cache_path, index=False)
    print(f'\nExtracted {len(df)} variants for {uniprot_id}. Cached → {cache_path}')
    return df


alpha_df = load_alphamissense(ALPHAMISSENSE_PATH, ALPHAMISSENSE_URL, ALPHAMISSENSE_UID)
print(alpha_df.describe())
print(alpha_df.head())


### 8a — Merge AlphaMissense as a model feature

`AlphaMissense_score` is merged into both `train_df` and `vus_df` and added to
`protected_features` so it is always retained during SHAP selection.

**Benchmark note (Section 11):** The enriched model has `AlphaMissense_score` as
*one feature among many*, plus ESM-2 delta embeddings, structural domain flags, and
biophysical scores. The benchmark answers: *"Does our pipeline add diagnostic value
on top of AlphaMissense?"* — it is not a pure head-to-head.


In [ ]:
# Left join — variants not in AlphaMissense receive NaN (imputed below)
train_df = train_df.merge(alpha_df[['Variant', 'AlphaMissense_score']], on='Variant', how='left')
vus_df   = vus_df.merge(alpha_df[['Variant', 'AlphaMissense_score']], on='Variant', how='left')

am_train_median = train_df['AlphaMissense_score'].median()
train_df['AlphaMissense_score'] = train_df['AlphaMissense_score'].fillna(am_train_median)
vus_df['AlphaMissense_score']   = vus_df['AlphaMissense_score'].fillna(am_train_median)

if 'AlphaMissense_score' not in protected_features:
    protected_features.append('AlphaMissense_score')

matched = alpha_df['Variant'].isin(train_df['Variant']).sum()
print(f'AlphaMissense merged. '
      f'{matched} / {len(train_df)} training variants matched directly.')


## Section 9 — Leakage-Safe 5-Fold Cross-Validation

**Single canonical CV loop. Do not re-run earlier CV drafts.**

Each fold:
1. **Grouped by residue position** — `StratifiedGroupKFold` ensures a variant at position 1004
   never appears in both train and validation.
2. **In-fold imputation** — medians computed on `X_tr` only, applied to `X_va`.
3. **In-fold SHAP feature selection** — preliminary XGB model trained on `X_tr`; mean |SHAP|
   selects top-30 features. Protected features always appended.
4. **BorderlineSMOTE** (borderline-1) — oversampling applied to `X_tr` only.
   Protected categorical/structural columns are sampled with replacement, never interpolated.
5. **In-fold isotonic calibration** — `CalibratedClassifierCV(FrozenEstimator(...), cv='prefit')`
   fitted on `X_va`; prevents the calibrator from seeing training data.
6. **MCC threshold optimisation** — 80 thresholds scanned (0.10 → 0.90) per fold.


In [ ]:
X_all = train_df.drop(columns=METADATA_DROP, errors='ignore').copy()
y_all = train_df['binary_label'].astype(int).copy()
X_all = preprocess_types(X_all, protected_features)

position_groups = train_df['Position'].astype(int)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

oof_probs          = np.zeros(len(train_df), dtype=float)
oof_fold           = np.full(len(train_df), -1, dtype=int)
fold_thresholds    = []
fold_sel_features  = []

rng = np.random.default_rng(RANDOM_STATE)

for fold, (tr_idx, va_idx) in enumerate(
        sgkf.split(X_all, y_all, groups=position_groups), start=1):
    print(f'\n===== Fold {fold} =====')
    X_tr, y_tr = X_all.iloc[tr_idx].copy(), y_all.iloc[tr_idx].copy()
    X_va, y_va = X_all.iloc[va_idx].copy(), y_all.iloc[va_idx].copy()

    # 1. In-fold imputation (fit on X_tr only)
    medians = fit_imputation_values(X_tr)
    X_tr    = apply_imputation(X_tr, medians, protected_features)
    X_va    = apply_imputation(X_va, medians, protected_features)

    # 2. SHAP-based feature selection on X_tr
    prelim = xgb.XGBClassifier(
        n_estimators=100, max_depth=4, tree_method='hist',
        enable_categorical=True, random_state=RANDOM_STATE, n_jobs=-1,
    )
    prelim.fit(X_tr, y_tr)
    explainer  = shap.TreeExplainer(prelim)
    shap_vals  = explainer.shap_values(X_tr)
    top30      = (
        pd.Series(np.abs(shap_vals).mean(axis=0), index=X_tr.columns)
        .sort_values(ascending=False).head(30).index.tolist()
    )
    selected = list(dict.fromkeys(top30 + protected_features))
    # keep only columns that actually exist in the dataframe
    selected = [f for f in selected if f in X_tr.columns]
    fold_sel_features.append(selected)

    # 3. BorderlineSMOTE on X_tr (protected cols sampled with replacement)
    smote_cols  = [c for c in selected if c not in protected_features]
    prot_cols   = [c for c in selected if c in protected_features]
    smote       = BorderlineSMOTE(random_state=RANDOM_STATE, kind='borderline-1')
    X_res_np, y_res = smote.fit_resample(X_tr[smote_cols].astype(float), y_tr)
    prot_rows   = X_tr[prot_cols].iloc[
        rng.integers(0, len(X_tr), len(y_res))
    ].reset_index(drop=True)
    X_res = pd.concat(
        [pd.DataFrame(X_res_np, columns=smote_cols), prot_rows], axis=1
    )[selected]

    # 4. Fold model on resampled data
    model_fold = xgb.XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05, tree_method='hist',
        enable_categorical=True, random_state=RANDOM_STATE, n_jobs=-1,
    )
    model_fold.fit(X_res, y_res)

    # 5. In-fold isotonic calibration on validation set
    calib_fold = CalibratedClassifierCV(
        FrozenEstimator(model_fold), method='isotonic', cv='prefit'
    )
    calib_fold.fit(X_va[selected], y_va)

    fold_probs        = calib_fold.predict_proba(X_va[selected])[:, 1]
    oof_probs[va_idx] = fold_probs
    oof_fold[va_idx]  = fold

    # 6. MCC threshold optimisation
    thresholds = np.linspace(0.10, 0.90, 80)
    mcc_scores = [matthews_corrcoef(y_va, (fold_probs >= t).astype(int))
                  for t in thresholds]
    best_thr   = float(thresholds[np.argmax(mcc_scores)])
    fold_thresholds.append(best_thr)
    print(f'Fold {fold} | MCC={max(mcc_scores):.4f} @ thr={best_thr:.3f} | '
          f'AUROC={roc_auc_score(y_va, fold_probs):.4f}')

print(f'\n=== OOF Summary ===')
print(f'AUROC    : {roc_auc_score(y_all, oof_probs):.4f}')
mean_threshold = float(np.mean(fold_thresholds))
print(f'MCC      : {matthews_corrcoef(y_all, (oof_probs >= mean_threshold).astype(int)):.4f}')
print(f'Threshold: {mean_threshold:.4f}  (mean of {len(fold_thresholds)} fold optima)')


## Section 10 — Feature Stability Analysis

In [ ]:
feature_counts   = Counter(f for ff in fold_sel_features for f in ff)
stable_features  = [f for f, c in feature_counts.items() if c >= 3]
# Always include protected features that are present in X_all
for p in protected_features:
    if p not in stable_features and p in X_all.columns:
        stable_features.append(p)

feature_stability = (
    pd.DataFrame({'feature': list(feature_counts.keys()),
                  'count':   list(feature_counts.values())})
    .sort_values(['count', 'feature'], ascending=[False, True])
    .reset_index(drop=True)
)
feature_stability['selected_ge_3'] = feature_stability['count'] >= 3
feature_stability.to_csv(FEATURE_STABILITY_PATH, index=False)

print(f'Stable feature count : {len(stable_features)}')
print(f'Saved                : {FEATURE_STABILITY_PATH}')

plt.figure(figsize=(10, 6))
feature_stability.head(25).plot(
    kind='barh', x='feature', y='count', color='steelblue', legend=False, ax=plt.gca()
)
plt.title('Top 25 Features by Selection Frequency across 5-Fold CV')
plt.xlabel('Number of folds selected (out of 5)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## Section 11 — Final Model Training & Artefact Export

In [ ]:
X_final      = X_all[stable_features].copy()
final_medians = fit_imputation_values(X_final)
X_final      = apply_imputation(X_final, final_medians, protected_features)

final_model = xgb.XGBClassifier(
    n_estimators=700,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    objective='binary:logistic',
    tree_method='hist',
    enable_categorical=True,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
final_model.fit(X_final, y_all)

# OOF isotonic calibrator — trained on out-of-fold probabilities vs true labels
isotonic_calibrator = IsotonicRegression(out_of_bounds='clip')
isotonic_calibrator.fit(oof_probs, y_all)
oof_probs_cal = isotonic_calibrator.predict(oof_probs)

# ── Export artefacts ──────────────────────────────────────────────────────
final_model.save_model(str(MODEL_PATH))
with open(CALIBRATOR_PATH, 'wb') as f:
    pickle.dump(isotonic_calibrator, f)
with open(FEATURES_PATH, 'w') as f:
    json.dump(stable_features, f, indent=2)
with open(THRESHOLD_PATH, 'w') as f:
    json.dump({'mean_threshold': mean_threshold,
               'fold_thresholds': fold_thresholds}, f, indent=2)

oof_df = train_df[['Variant', 'binary_label']].copy()
oof_df['fold']               = oof_fold
oof_df['oof_prob']           = oof_probs
oof_df['oof_prob_calibrated'] = oof_probs_cal
oof_df['pred_at_mean_thr']   = (oof_probs_cal >= mean_threshold).astype(int)
oof_df.to_csv(OOF_PATH, index=False)

auroc = roc_auc_score(y_all, oof_probs_cal)
mcc   = matthews_corrcoef(y_all, (oof_probs_cal >= mean_threshold).astype(int))
brier = brier_score_loss(y_all, oof_probs_cal)

print('=== Final OOF Validation Metrics ===')
print(f'AUROC            : {auroc:.4f}')
print(f'MCC (@{mean_threshold:.3f})   : {mcc:.4f}')
print(f'Brier score      : {brier:.4f}  (lower is better; 0 = perfect)')
print(f'\nModel saved      → {MODEL_PATH}')
print(f'Calibrator saved → {CALIBRATOR_PATH}')
print(f'OOF predictions  → {OOF_PATH}')


## Section 12 — AlphaMissense Benchmark

Compares our enriched model (which includes `AlphaMissense_score` as one feature)
against AlphaMissense alone on the OOF predictions.

The published pathogenicity threshold from Cheng et al. (2023) is **0.564**.
We use both the paper threshold and a simple 0.5 cutoff for comparison.


In [ ]:
bench_df = oof_df.merge(alpha_df, on='Variant', how='inner')
bench_df['AlphaMissense_score'] = pd.to_numeric(
    bench_df['AlphaMissense_score'], errors='coerce'
)
bench_df = bench_df.dropna(subset=['AlphaMissense_score']).copy()

if bench_df.empty:
    print('WARNING: No overlap rows found for benchmark — check variant name format.')
else:
    model_auroc = roc_auc_score(bench_df['binary_label'], bench_df['oof_prob_calibrated'])
    alpha_auroc = roc_auc_score(bench_df['binary_label'], bench_df['AlphaMissense_score'])

    model_mcc   = matthews_corrcoef(
        bench_df['binary_label'],
        (bench_df['oof_prob_calibrated'] >= mean_threshold).astype(int)
    )
    alpha_mcc_05  = matthews_corrcoef(
        bench_df['binary_label'],
        (bench_df['AlphaMissense_score'] >= 0.5).astype(int)
    )
    ALPHAMISSENSE_PAPER_THRESHOLD = 0.564
    alpha_mcc_pub = matthews_corrcoef(
        bench_df['binary_label'],
        (bench_df['AlphaMissense_score'] >= ALPHAMISSENSE_PAPER_THRESHOLD).astype(int)
    )

    model_brier = brier_score_loss(bench_df['binary_label'], bench_df['oof_prob_calibrated'])
    alpha_brier = brier_score_loss(bench_df['binary_label'], bench_df['AlphaMissense_score'])

    print('=== AlphaMissense Benchmark (Source: DeepMind tsv.gz, Cheng et al. Science 2023) ===')
    print(f'Overlap variants : {len(bench_df)}')
    print()
    print(f'{"Metric":<30} {"Enriched model":>16} {"AlphaMissense":>16}')
    print('-' * 65)
    print(f'{"AUROC":<30} {model_auroc:>16.4f} {alpha_auroc:>16.4f}')
    print(f'{"MCC (optimised thr)":<30} {model_mcc:>16.4f} '
          f'{"(thr=0.50)":>6} {alpha_mcc_05:>9.4f}')
    print(f'{"MCC (paper thr 0.564)":<30} {"—":>16} {alpha_mcc_pub:>16.4f}')
    print(f'{"Brier score":<30} {model_brier:>16.4f} {alpha_brier:>16.4f}')
    print()
    print('Note: Enriched model includes AlphaMissense_score as one input feature.')


## Section 13 — VUS Predictions

In [ ]:
X_vus = vus_df.drop(columns=METADATA_DROP, errors='ignore').copy()
X_vus = preprocess_types(X_vus, protected_features)

# Align to training feature set (add any missing columns as NaN)
for f in stable_features:
    if f not in X_vus.columns:
        X_vus[f] = np.nan
X_vus = X_vus[stable_features]
X_vus = apply_imputation(X_vus, final_medians, protected_features)

vus_probs_raw = final_model.predict_proba(X_vus)[:, 1]
vus_probs_cal = isotonic_calibrator.predict(vus_probs_raw)

vus_out = vus_df[['Variant', 'Significance']].copy()
vus_out['P_pathogenic_raw']       = vus_probs_raw
vus_out['P_pathogenic_calibrated'] = vus_probs_cal
# Classification thresholds: high confidence only
vus_out['Classification'] = np.where(
    vus_probs_cal >= 0.90, 'Likely Pathogenic',
    np.where(vus_probs_cal <= 0.10, 'Likely Benign', 'Remain VUS'),
)

vus_out.to_csv(VUS_PRED_PATH, index=False)
print(f'VUS predictions saved → {VUS_PRED_PATH}')
print(vus_out['Classification'].value_counts())
display(vus_out.sort_values('P_pathogenic_calibrated', ascending=False).head(10))


## Section 14 — 2024 Gold Standard Validation

10 ABCA4 variants that were functionally characterised or reclassified in 2024.
We compare our model's calibrated probability against AlphaMissense.


In [ ]:
GOLD_STANDARD = [
    ('G145R',  1), ('A1038V', 1), ('L1091P', 1), ('G1453R', 1), ('G1570R', 1),
    ('L1939P', 1), ('G2115E', 1), ('G2159R', 1), ('I2151V', 0), ('P1455L', 0),
]
gold_variants = [v for v, _ in GOLD_STANDARD]
gold_labels   = {v: l for v, l in GOLD_STANDARD}

# Build full lookup including train + VUS predictions
full_df = pd.concat([train_df, vus_df], axis=0).drop_duplicates(subset=['Variant'])
X_full  = full_df.drop(columns=METADATA_DROP, errors='ignore').copy()
X_full  = preprocess_types(X_full, protected_features)
for f in stable_features:
    if f not in X_full.columns:
        X_full[f] = np.nan
X_full = X_full[stable_features]
X_full = apply_imputation(X_full, final_medians, protected_features)

full_probs_raw = final_model.predict_proba(X_full)[:, 1]
full_probs_cal = isotonic_calibrator.predict(full_probs_raw)

lookup = full_df[['Variant', 'Position']].copy()
lookup['Model_Score'] = full_probs_cal
if 'AlphaMissense_score' in full_df.columns:
    lookup['AlphaMissense_score'] = full_df['AlphaMissense_score'].values

summary_rows = []
for var, true_label in GOLD_STANDARD:
    match = lookup[lookup['Variant'] == var]
    # Fallback: match by numeric position extracted from variant string
    if match.empty:
        m = re.search(r'\d+', var)
        if m:
            pos_int = int(m.group())
            match = lookup[lookup['Position'].astype(int) == pos_int]
    row = {'Variant': var, 'True_Label': true_label,
           'Found_In_Dataset': not match.empty}
    if not match.empty:
        r = match.iloc[0]
        row['Model_Score']        = round(r['Model_Score'], 4)
        row['AlphaMissense_score'] = round(r.get('AlphaMissense_score', float('nan')), 4)
        row['Model_Pred']  = 1 if r['Model_Score'] >= mean_threshold else 0
        row['AM_Pred']     = 1 if r.get('AlphaMissense_score', 0) >= 0.564 else 0
        row['Model_Correct'] = int(row['Model_Pred'] == true_label)
        row['AM_Correct']    = int(row['AM_Pred']    == true_label)
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
print('=== 2024 Gold Standard Validation ===')
display(summary)
found = summary[summary['Found_In_Dataset']]
if len(found) > 0:
    print(f'\nFound {len(found)}/10 gold standard variants in dataset.')
    print(f'Model accuracy  : {found["Model_Correct"].mean():.0%}')
    print(f'AlphaMissense   : {found["AM_Correct"].mean():.0%}')


## Section 15 — Leakage & Robustness Audit

In [ ]:
def run_robustness_audit(train_df_, vus_df_, stable_feats, X_all_):
    """Systematic check for data leakage, distribution shift, and feature integrity."""
    from scipy.stats import ks_2samp
    results = {}

    # 1. Variant overlap between train and VUS
    train_vars = set(train_df_['Variant'])
    vus_vars   = set(vus_df_['Variant'])
    overlap_vars = train_vars & vus_vars
    results['variant_overlap_train_vs_vus'] = len(overlap_vars)

    # 2. Position overlap (spatial leakage check)
    train_pos = set(train_df_['Position'].astype(int))
    vus_pos   = set(vus_df_['Position'].astype(int))
    results['position_overlap_train_vs_vus'] = len(train_pos & vus_pos)

    # 3. Target column leakage in X_all
    leak_cols = [c for c in X_all_.columns
                 if 'label' in c.lower() or 'significance' in c.lower()]
    results['target_columns_in_X_all'] = leak_cols

    # 4. Calibration probability range validity
    valid_range = bool((oof_probs_cal >= 0).all() and (oof_probs_cal <= 1).all())
    results['oof_calibrated_range_valid'] = valid_range

    # 5. Feature distribution shift (KS test train vs VUS)
    key_features = ['AlphaMissense_score', 'esm_pca_00', 'Consurf_score']
    shifts = {}
    for f in key_features:
        if f in train_df_.columns and f in vus_df_.columns:
            stat, p = ks_2samp(
                train_df_[f].dropna().values,
                vus_df_[f].dropna().values,
            )
            shifts[f] = {'ks_stat': round(float(stat), 4),
                         'p_value': round(float(p), 4),
                         'significant_shift': bool(p < 0.05)}
    results['feature_distribution_shift'] = shifts

    # 6. Missing value counts in stable feature set
    train_nan = int(train_df_[[f for f in stable_feats if f in train_df_.columns]].isna().sum().sum())
    vus_nan   = int(vus_df_  [[f for f in stable_feats if f in vus_df_.columns  ]].isna().sum().sum())
    results['nan_in_stable_features_train'] = train_nan
    results['nan_in_stable_features_vus']   = vus_nan

    # 7. Class balance
    results['class_balance_train'] = train_df_['binary_label'].value_counts(normalize=True).to_dict()

    return results


audit = run_robustness_audit(train_df, vus_df, stable_features, X_all)
print(json.dumps(audit, indent=2, default=str))

# Assertions — hard failures indicate a misconfigured pipeline
assert audit['target_columns_in_X_all'] == [], (
    f'LEAKAGE: Target columns found in X_all: {audit["target_columns_in_X_all"]}'
)
assert audit['oof_calibrated_range_valid'], 'Calibrated probabilities out of [0,1] range!'
print('\nAll leakage assertions passed.')
